# Enclave Evaluation — Model Owner

| Actor | Email | Role |
|-------|-------|------|
| **Enclave** | `enclave@ai-safety.org` | Trusted execution environment (sealed memory) |
| **Model Owner** | (this notebook) | Owns the private language-model weights (Qwen) |
| **Benchmark Owner** | `benchmark_owner@ai-safety.org` | Owns the private Vietnamese-harm benchmark |
| **Researcher** | `researcher@ai-safety.org` | Writes the eval code, submits the job |

This notebook drives only the **Model Owner** steps; the Benchmark Owner and Researcher run their own notebooks in parallel.
Self-contained: weights are pulled from Hugging Face, `infer.py` from the public repo.

In [ ]:
!uv pip install -Uq huggingface-hub "git+https://github.com/OpenMined/PySyft.git@dev#subdirectory=packages/syft-enclave"

## Setup

In [ ]:
import os
from pathlib import Path

import requests

os.environ["PRE_SYNC"] = "false"

from syft_enclaves import login_do

# This notebook is self-contained — no local checkout of blindfold needed. Code + benchmark
# are fetched from the public repo; weights come from Hugging Face. If the repo moves to the
# OpenMined org, change GITHUB_REPO. Pin GITHUB_REF to a commit/tag for a reproducible demo.
GITHUB_REPO = "khoaguin/blindfold"
GITHUB_REF = "main"
RAW = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_REF}"

# Downloads land here — next to the notebook (in Colab: /content/...) so the fetched code +
# benchmark show up in the Files pane and can be opened live during the demo.
BASE = Path.cwd()


def fetch(repo_path: str, dest: Path) -> Path:
    # Download one file from the public repo into `dest`.
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f"{RAW}/{repo_path}"
    resp = requests.get(url, timeout=30)
    if not resp.ok:
        raise RuntimeError(f"could not fetch {url} (HTTP {resp.status_code}) — is {GITHUB_REPO} public yet?")
    dest.write_bytes(resp.content)
    return dest

In [ ]:
ENCLAVE_EMAIL     = "enclave@ai-safety.org"
MODEL_OWNER_EMAIL = "model_owner@ai-safety.org"
RESEARCHER_EMAIL  = "researcher@ai-safety.org"

print(f"  Enclave: {ENCLAVE_EMAIL}  |  Researcher: {RESEARCHER_EMAIL}")

## Step 0 — Log in as Model Owner

In [ ]:
model_owner = login_do(MODEL_OWNER_EMAIL)
print(f"  Model Owner : {model_owner.email}")

In [ ]:
# Optionally clear state for a fresh run
model_owner._manager.delete_syftbox()
model_owner._manager.peer_manager.write_own_version()

### Launch the enclave

Start the enclave service for `enclave@ai-safety.org` out-of-band. It runs the job
automatically once **both** owners approve — no notebook drives it.

## Step 1 — Peer with the Enclave

In [ ]:
model_owner.add_peer(ENCLAVE_EMAIL)
model_owner.sync()
print(f"  Model Owner peered with enclave ({ENCLAVE_EMAIL})")

### Step 1.1 — Wait for the Researcher peer request, then approve

The Researcher notebook adds you as a peer. Re-run the cell below until their request appears, then approve.

In [ ]:
model_owner.sync()
model_owner.peers

In [ ]:
model_owner.approve_peer_request(RESEARCHER_EMAIL, peer_must_exist=False)
model_owner.sync()
print("  Researcher peer approved")

### Step 1.2 — Attest enclave's identity

In [ ]:
# Wait for the enclave to accept the peer request, then attest
model_owner.attest_peer(ENCLAVE_EMAIL)

## Step 2.1 — Model assets

Weights download from Hugging Face into **`./models/qwen2.5-0.5b/`** (visible in the Files pane;
cached, so re-runs are instant). The lab's real `infer.py` is fetched there too — it ships
**inside** the private dataset. The public mock is the stub `infer.py` in **`./mock_model/`**.

In [ ]:
from huggingface_hub import snapshot_download

MODEL = "qwen2.5-0.5b"  # the model under audit
HF_REPO = "mlx-community/Qwen2.5-0.5B-Instruct-4bit"

# Private weights → ./models/<model>/ (visible; snapshot_download is cached → instant re-runs).
MODEL_DIR = BASE / "models" / MODEL
snapshot_download(
    repo_id=HF_REPO,
    local_dir=str(MODEL_DIR),
    ignore_patterns=["*.pt", "*.pth", "*.h5", "*.msgpack", "*.onnx", "*.tflite", "*.bin"],
)
# The lab's real infer.py ships inside the private dataset, alongside the weights (their IP).
fetch("code/model_owner_code/infer.py", MODEL_DIR / "infer.py")


def fetch_mock_model() -> Path:
    # Public mock: the stub infer.py (same init/generate interface, no weights) → ./mock_model/.
    d = BASE / "mock_model"
    fetch("code/model_owner_code/mock_infer.py", d / "infer.py")
    return d


print(f"  weights + infer.py -> {MODEL_DIR}")

## Step 2.2 — Upload the model weights

* **mock**: a stub `infer.py` (no weights) — the researcher develops against this
* **private**: real Qwen weights + the model owner's `infer.py` — only the enclave receives this

In [ ]:
mock_dir = fetch_mock_model()
_gb = sum(p.stat().st_size for p in MODEL_DIR.rglob("*") if p.is_file()) / 1e9

model_owner.create_dataset(
    name="model",
    mock_path=mock_dir,
    private_path=str(MODEL_DIR),
    summary="Qwen model weights (mlx safetensors) + infer.py — private to the enclave",
    users=[RESEARCHER_EMAIL, ENCLAVE_EMAIL],
    upload_private=True,
    sync=False,
)
model_owner.share_private_dataset("model", ENCLAVE_EMAIL)
model_owner.sync()
print(f"  Model Owner uploaded 'model'  (private: {_gb:.2f} GB)")

## Step 3 — Wait for the Researcher to submit the job, then approve

The Researcher submits `vn_harm_audit` to the enclave. Re-sync until it appears, inspect the
submitted code, then approve. The enclave only runs once **both** owners approve.

In [ ]:
JOB_NAME = "vn_harm_audit"
model_owner.sync()
model_owner_job = model_owner.jobs[JOB_NAME]
print(f"  Model Owner sees '{JOB_NAME}'  status={model_owner_job.status}")
model_owner_job  # inspect the submitted code before approving

In [ ]:
model_owner.approve_job(model_owner_job)
model_owner.sync()
print("  Model Owner approved")

## Step 4 — Receive results

Results arrive here because the Researcher submitted with `share_results_with_do=True`.

In [ ]:
model_owner.sync()
model_owner_job = model_owner.jobs[JOB_NAME]
print(f"  Output files : {model_owner_job.output_paths}")
assert len(model_owner_job.output_paths) > 0